In [5]:
import argparse
import os.path as op
from numrisk.utils.data import Subject
from nilearn import surface
import nibabel as nb
#from numrisk.fmri_analysis.encoding_model.fit_nprf import get_key_target_dir
from tqdm import tqdm
from nipype.interfaces.freesurfer import SurfaceTransform

# Parameters
subject_id = 1  
session = 1  
bids_folder = '/mnt_03/ds-dnumrisk'

sub = Subject(subject_id, bids_folder=bids_folder)
subject = f'{int(subject_id):02d}'
target_dir = op.join(bids_folder, 'derivatives', 'glm_stim1.denoise', f'sub-{subject}', f'ses-{session}', 'func')


In [6]:

smoothed = False  
denoise = True  

# Initialize subject and paths


surfinfo = sub.get_surf_info_fs()

par_keys = ['mu', 'sd', 'amplitude', 'baseline', 'r2']  # Parameters to process

prf_pars_volume = sub.get_prf_parameters_volume(session, smoothed=smoothed, denoise=denoise, keys=par_keys, return_image=True, cross_validated=False)

# Define target directory
dir = 'glm_stim1'
if denoise:
    dir += '.denoise'
target_dir = op.join(bids_folder, 'derivatives', dir, f'sub-{subject}', f'ses-{session}', 'func')

# Loop through hemispheres and parameters
for hemi in ['L', 'R']:
    samples = surface.vol_to_surf(prf_pars_volume, surfinfo[hemi]['outer'], inner_mesh=surfinfo[hemi]['inner'])
    fs_hemi = 'lh' if hemi == 'L' else 'rh'



KeyboardInterrupt: 

In [14]:
prf_pars_volume

In [ ]:
for ix, par in enumerate(par_keys):
    im = nb.gifti.GiftiImage(darrays=[nb.gifti.GiftiDataArray(samples[:, ix].astype('float32'))]) #added the as.type('float32') to avoid error
    target_fn = op.join(target_dir, f'sub-{subject}_ses-{session}_desc-{par}.optim.nilearn_space-fsnative_hemi-{hemi}.func.gii')
    nb.save(im, target_fn)

    # Transform to fsaverage space
    subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')
    sxfm = SurfaceTransform(subjects_dir=subjects_dir)
    sxfm.inputs.source_file = target_fn
    sxfm.inputs.out_file = target_fn.replace('fsnative', 'fsaverage5')
    sxfm.inputs.source_subject = f'sub-{subject}'
    sxfm.inputs.target_subject = 'fsaverage'
    sxfm.inputs.hemi = fs_hemi

    r = sxfm.run()

In [27]:
import nibabel as nb

subject = '01'
session = '1'

fn = op.join(target_dir, f'sub-{subject}_ses-{session}_task-magjudge_space-T1w_desc-stims1_pe.nii.gz')
betas_vol = nb.load(fn)

from nilearn.maskers import NiftiMasker
mask = sub.get_brain_mask()
masker = NiftiMasker(mask_img=mask)

In [28]:
surfinfo = sub.get_surf_info_fs()


In [29]:
betas = masker.fit_transform(betas_vol)
betas_it= masker.inverse_transform(betas)
hemi= 'L'#, 'R']:
samples = surface.vol_to_surf(betas_it, surfinfo[hemi]['outer'], inner_mesh=surfinfo[hemi]['inner'])
fs_hemi = 'lh' if hemi == 'L' else 'rh'
im = nb.gifti.GiftiImage(darrays=[nb.gifti.GiftiDataArray(samples.astype('float32'))]) #added the as.type('float32') to avoid error

print(samples.shape)

(145302, 180)


In [46]:
sub='02'
target_dir = op.join(bids_folder, 'derivatives', 'glm_stim1.denoise', f'sub-{sub}', f'ses-1', 'func')
target_fn = op.join(target_dir, f'sub-{sub}_ses-1_task-magjudge_space-fsnative_stim-1_hemi-L.func.gii')

samples = nib.load(target_fn).agg_data()
print(samples.shape)

(131917, 180)


In [39]:
darrays = [nb.gifti.GiftiDataArray(samples[:, ix].astype('float32')) for ix in range(samples.shape[1])]
im = nb.gifti.GiftiImage(darrays=darrays) #added the as.type('float32') to avoid error


In [ ]:
for da in im.darrays:
    da.intent = nb.nifti1.intent_codes['NIFTI_INTENT_TIME_SERIES']

nb.save(im, target_fn)

In [42]:
im.darrays

[<GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time series[145302]>,
 <GiftiDataArray time ser

In [10]:
target_fn = op.join(target_dir, 'sub-01_ses-1_task-magjudge_space-fsnative_stim-1_hemi-L.func.gii')

In [43]:
# Transform to fsaverage space
subjects_dir = op.join(bids_folder, 'derivatives', 'freesurfer')
sxfm = SurfaceTransform(subjects_dir=subjects_dir)
sxfm.inputs.source_file = target_fn
sxfm.inputs.out_file = target_fn.replace('fsnative', 'fsaverage5')
sxfm.inputs.source_subject = f'sub-{subject}'
sxfm.inputs.target_subject = 'fsaverage5'
sxfm.inputs.hemi = fs_hemi

r = sxfm.run()

250403-08:28:14,610 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.610820:** DA[0] has coordsys with intent NIFTI_INTENT_TIME_SERIES (should be NIFTI_INTENT_POINTSET)
250403-08:28:14,616 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.616302:** DA[1] has coordsys with intent NIFTI_INTENT_TIME_SERIES (should be NIFTI_INTENT_POINTSET)
250403-08:28:14,621 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.621724:** DA[2] has coordsys with intent NIFTI_INTENT_TIME_SERIES (should be NIFTI_INTENT_POINTSET)
250403-08:28:14,627 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.627301:** DA[3] has coordsys with intent NIFTI_INTENT_TIME_SERIES (should be NIFTI_INTENT_POINTSET)
250403-08:28:14,632 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.632806:** DA[4] has coordsys with intent NIFTI_INTENT_TIME_SERIES (should be NIFTI_INTENT_POINTSET)
250403-08:28:14,638 nipype.interface INFO:
	 stderr 2025-04-03T08:28:14.638420:** DA[5] has coordsys with intent NIFTI_INTENT_TIME_SERIES (s

In [44]:
import nibabel as nib


# Load the output file and print its shape
output_file = sxfm.inputs.out_file
output_data = nib.load(output_file)
print(f"Output file shape: {output_data.darrays[0].data.shape}")

Output file shape: (10242,)


In [17]:
sxfm.inputs.out_file

'/mnt_03/ds-dnumrisk/derivatives/glm_stim1.denoise/sub-01/ses-1/func/sub-01_ses-1_task-magjudge_space-fsaverage_stim-1_hemi-L.func.gii'

In [45]:
fn = sxfm.inputs.out_file
im = nb.load(fn)
im.darrays

[<GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataArray time series[10242]>,
 <GiftiDataA

In [25]:
 (20484 /2 )

10242.0

In [16]:
output_data.darrays[0].data.shape

(163842,)

In [31]:
betas = masker.fit_transform(betas_vol)
betas_it= masker.inverse_transform(betas)

for hemi in ['L', 'R']:
    samples = surface.vol_to_surf(betas_it, surfinfo[hemi]['outer'], inner_mesh=surfinfo[hemi]['inner'])
    fs_hemi = 'lh' if hemi == 'L' else 'rh'

    im = nb.gifti.GiftiImage(darrays=[nb.gifti.GiftiDataArray(samples.astype('float32'))]) #added the as.type('float32') to avoid error
    target_fn = op.join(target_dir, f'sub-{subject}_ses-{session}_task-magjudge_space-T1w_desc-stims1_hemi-{hemi}.func.gii')
    nb.save(im, target_fn)

In [ ]:
data = betas_it.get_fdata()

# Check the shape of the data
print(data.shape)  # Example: (X, Y, Z, T) where T is the number of volumes

# Extract a single volume (e.g., the first volume)
single_volume = data[..., 0]

# Create a new NIfTI image for the single volume
single_volume_img = nb.Nifti1Image(single_volume, affine=betas_it.affine)


(59, 71, 49, 180)


In [ ]:
data = betas_it.get_fdata()

# Check the shape of the data
print(data.shape)  # Example: (X, Y, Z, T) where T is the number of volumes

# Extract a single volume (e.g., the first volume)
single_volume = data[..., 0]

# Create a new NIfTI image for the single volume
single_volume_img = nb.Nifti1Image(single_volume, affine=betas_it.affine)


In [42]:
# trying to image stuff

from nilearn.plotting import plot_surf_stat_map
from nilearn.datasets import load_fsaverage
import numpy as np
from nilearn.datasets import load_fsaverage_data

subject = '01'
session = '1'
stim = '1'
hemi = 'L'
key = f'glm_stim{stim}.denoise'

target_dir = op.join(bids_folder, 'derivatives', key , f'sub-{subject}', f'ses-{session}', 'func')

surface_image = nb.load(op.join(target_dir, f'sub-{subject}_ses-{session}_task-magjudge_space-fsaverage5_stim-{stim}_hemi-{hemi}.func.gii'))

# Extract a single volume (e.g., the first volume)
single_volume = surface_image[..., 0]

# Create a new NIfTI image for the single volume
single_volume_img = nb.Nifti1Image(single_volume, affine=betas_it.affine)

curv_sign = load_fsaverage_data(data_type="curvature")
for hemi, data in curv_sign.data.parts.items():
    curv_sign.data.parts[hemi] = np.sign(data)

fsaverage_meshes = load_fsaverage()

# In this example we will only plot the right hemisphere
hemi = "right"

fig = plot_surf_stat_map(
    stat_map=surface_image,
    surf_mesh=fsaverage_meshes["inflated"],
    hemi=hemi,
    title="Surface with matplotlib",
    colorbar=True,
    threshold=1.0,
    bg_map=curv_sign,
)
fig.show()

TypeError: Cannot slice image objects.